# Module 2 — Topic 6: Data Privacy & Protection
## Live Demo Notebook

**Scenario:** The startup's CEO has approved sharing a performance analysis with an external logistics partner to improve delivery times. The partner needs order-level data — but they must NOT receive any customer personal information.

Your task:
1. Identify every PII column in `orders_clean.csv`
2. Apply appropriate protection (masking, hashing, removal)
3. Produce a privacy-safe export that the logistics partner can use
4. Produce a separate aggregated summary for public reporting

**This is non-negotiable — sharing unprotected customer data with a third party violates the NDPA 2023.**

---

## Part 1 — Load and Audit the Dataset for PII

In [ ]:
import pandas as pd
import hashlib

df = pd.read_csv("../../../data/orders_clean.csv", encoding="utf-8-sig",
                 parse_dates=["order_date"])

print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print()
print("All columns:")
for i, col in enumerate(df.columns):
    print(f"  {i:>2}. {col}")

In [ ]:
# PII AUDIT — classify every column
pii_classification = {
    "order_id":          "Not PII — system-generated ID",
    "customer_name":     "DIRECT PII — identifies the person",
    "phone":             "DIRECT PII — contact detail, linked to BVN",
    "email":             "DIRECT PII — contact detail, account access vector",
    "state":             "Indirect — location (state level, acceptable)",
    "city":              "Indirect — location (city level, borderline)",
    "product_category":  "Not PII — aggregated category",
    "product_name":      "Not PII — product identifier",
    "quantity":          "Not PII — transaction attribute",
    "unit_price":        "Not PII — transaction attribute",
    "total_amount":      "Not PII — transaction attribute",
    "order_date":        "Indirect — behaviour pattern (date of purchase)",
    "delivery_status":   "Not PII — operational status",
    "payment_method":    "Not PII — financial behaviour (category only)",
    "order_month":       "Not PII — derived from date",
    "order_year":        "Not PII — derived from date",
}

print("===== PII Audit =====")
for col, classification in pii_classification.items():
    if col in df.columns:
        flag = "[PII]" if "PII" in classification else "     "
        print(f"  {flag} {col:<20}: {classification}")

In [ ]:
# Show what the PII columns look like — so students understand what's at risk
print("Sample of PII columns (DO NOT SHARE THIS):")
df[["customer_name", "phone", "email"]].head(8)

---
## Part 2 — Masking: Obscure but Keep Column Shape

In [ ]:
# Name masking — keep initial only
def mask_name(name):
    """Keep first initial, mask surname with asterisks."""
    parts = str(name).strip().split()
    if len(parts) >= 2:
        return parts[0][0] + ". " + "*" * len(parts[-1])
    return parts[0][0] + "***"

df["customer_masked"] = df["customer_name"].apply(mask_name)

print("Name masking:")
df[["customer_name", "customer_masked"]].head(8)

In [ ]:
# Phone masking — show last 4 digits only (standard banking practice)
def mask_phone(phone):
    if pd.isna(phone) or phone == "Not provided":
        return phone
    phone_str = str(phone)
    return "*" * (len(phone_str) - 4) + phone_str[-4:]

df["phone_masked"] = df["phone"].apply(mask_phone)

print("Phone masking:")
df[["phone", "phone_masked"]].head(8)

In [ ]:
# Email masking — hide local part, keep domain
def mask_email(email):
    if pd.isna(email) or "@" not in str(email):
        return email
    local, domain = email.split("@", 1)
    return local[:2] + "***@" + domain

df["email_masked"] = df["email"].apply(mask_email)

print("Email masking:")
df[["email", "email_masked"]].head(8)

---
## Part 3 — Hashing: Consistent Pseudonymisation

In [ ]:
# Hashing with a salt — one-way, consistent, cannot be reversed without the salt
SALT = "ndpa_compliance_2024"   # In production: store this securely, never hard-code

def hash_pii(value, salt=SALT):
    """
    Produce a 16-character pseudonym from a personal value.
    Same input → same output (consistent for grouping).
    Cannot be reversed without the salt.
    """
    combined = str(value) + salt
    return hashlib.sha256(combined.encode()).hexdigest()[:16]

# Hash customer_name to create a reusable customer_id
df["customer_id"] = df["customer_name"].apply(hash_pii)

print("Hashed customer IDs:")
df[["customer_name", "customer_id"]].head(8)

In [ ]:
# Verify: same customer name always produces same hash
test_name = df["customer_name"].iloc[0]
hash1 = hash_pii(test_name)
hash2 = hash_pii(test_name)
hash3 = hash_pii(test_name)

print(f"Name: '{test_name}'")
print(f"Hash 1: {hash1}")
print(f"Hash 2: {hash2}")
print(f"Hash 3: {hash3}")
print(f"All identical: {hash1 == hash2 == hash3}")
print()
print("This means: a customer's orders across different datasets")
print("can still be linked by customer_id — without exposing their name.")

In [ ]:
# Also hash phone numbers — useful for cross-dataset linking
df["phone_id"] = df["phone"].apply(
    lambda x: hash_pii(x) if x != "Not provided" else "Not provided"
)

print("Phone hashing:")
df[["phone", "phone_id"]].head(6)

---
## Part 4 — Building the Privacy-Safe Export

In [ ]:
# Columns to DROP — direct PII that must never leave the secure environment
DROP_COLUMNS = ["customer_name", "phone", "email",
                "customer_masked", "phone_masked", "email_masked",
                "phone_id"]  # phone_id included — logistics partner doesn't need phone linkage

# Build the safe DataFrame
df_safe = df.drop(
    columns=[col for col in DROP_COLUMNS if col in df.columns]
).copy()

print(f"Original columns: {df.shape[1]}")
print(f"Safe export columns: {df_safe.shape[1]}")
print()
print("Remaining columns:")
for col in df_safe.columns:
    print(f"  - {col}")

In [ ]:
# VERIFICATION — confirm no direct PII remains
direct_pii = ["customer_name", "phone", "email"]
remaining_pii = [col for col in direct_pii if col in df_safe.columns]

print("===== PII Verification =====")
if len(remaining_pii) == 0:
    print("Direct PII columns: NONE FOUND")
    print("customer_id (hash) present:", "customer_id" in df_safe.columns)
    print("Safe to share with logistics partner.")
else:
    print(f"WARNING: PII still present in columns: {remaining_pii}")
    print("DO NOT EXPORT until these are removed.")
print()
print("Sample of safe export:")
df_safe[["order_id", "customer_id", "state", "city", "product_category",
          "total_amount", "delivery_status", "order_date"]].head(6)

In [ ]:
# Save the privacy-safe export
df_safe.to_csv("../../../data/orders_safe_export.csv",
               index=False, encoding="utf-8-sig")

print(f"orders_safe_export.csv saved: {df_safe.shape[0]} rows x {df_safe.shape[1]} columns")
print("This file is safe to share with the logistics partner.")

---
## Part 5 — Aggregated Summary for Public Reporting

In [ ]:
# State-level summary — completely anonymous, safe for public reports
state_summary = (
    df_safe.groupby("state")
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("total_amount", "sum"),
        avg_order=("total_amount", "mean"),
        delivery_rate=("delivery_status",
                       lambda x: (x == "Delivered").mean() * 100)
    )
    .round({"total_revenue": 0, "avg_order": 0, "delivery_rate": 1})
    .sort_values("total_revenue", ascending=False)
)

print("===== State Performance Summary (Public-Safe) =====")
state_summary

In [ ]:
# Apply the n < 5 rule — suppress any group with fewer than 5 orders
before_count = len(state_summary)
state_summary_safe = state_summary[state_summary["order_count"] >= 5].copy()
after_count = len(state_summary_safe)

print(f"Groups before n<5 filter: {before_count}")
print(f"Groups after  n<5 filter: {after_count}")
if before_count > after_count:
    suppressed = before_count - after_count
    print(f"Suppressed: {suppressed} groups with fewer than 5 orders")
else:
    print("All groups have 5+ orders — none suppressed.")
print()
print("Safe state summary (n>=5):")
state_summary_safe

In [ ]:
# Category summary for public reporting
category_summary = (
    df_safe.groupby("product_category")
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("total_amount", "sum"),
        avg_order=("total_amount", "mean"),
    )
    .round(0)
    .sort_values("total_revenue", ascending=False)
)

print("===== Category Summary (Public-Safe) =====")
category_summary

In [ ]:
# Monthly trend — useful for logistics planning, completely anonymous
monthly_trend = (
    df_safe.groupby("order_month")
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("total_amount", "sum"),
    )
    .round(0)
    .sort_index()
)

print("===== Monthly Order Trend (Public-Safe) =====")
monthly_trend

---
## Part 6 — Save Public Summary

In [ ]:
# Save aggregated summaries for the public report
state_summary_safe.to_csv(
    "../../../data/summary_by_state.csv",
    encoding="utf-8-sig"
)
category_summary.to_csv(
    "../../../data/summary_by_category.csv",
    encoding="utf-8-sig"
)
monthly_trend.to_csv(
    "../../../data/summary_monthly.csv",
    encoding="utf-8-sig"
)

print("Public summary files saved:")
print("  summary_by_state.csv    — state-level performance")
print("  summary_by_category.csv — category-level performance")
print("  summary_monthly.csv     — monthly trend")
print()
print("These files contain NO personal data and are safe for:")
print("  - External partner sharing")
print("  - Public dashboards")
print("  - Investor reports")
print("  - Press releases")

---
## Part 7 — Final Checklist and Data File Inventory

In [ ]:
print("===== DATA FILE INVENTORY =====")
print()
print("SECURE ENVIRONMENT ONLY (never share externally):")
print("  orders.csv              — raw, unclean, contains full PII")
print("  orders_clean.csv        — clean, contains full PII")
print()
print("SAFE FOR LOGISTICS PARTNER (pseudonymised, order-level):")
print("  orders_safe_export.csv  — no names/phone/email, customer_id hash only")
print()
print("SAFE FOR PUBLIC (fully aggregated, no individuals visible):")
print("  summary_by_state.csv")
print("  summary_by_category.csv")
print("  summary_monthly.csv")
print()
print("===== PRIVACY DECISIONS LOG =====")
print()
print("1. customer_name → HASHED to customer_id (SHA256 + salt)")
print("   Reason: logistics partner needs to link orders to same customer")
print("           without seeing the customer's real name")
print()
print("2. phone, email → DROPPED entirely from logistics export")
print("   Reason: logistics partner has no legitimate need for contact details")
print()
print("3. Public summaries → AGGREGATED with n>=5 filter")
print("   Reason: prevents re-identification of individuals in small groups")
print()
print("4. city field → RETAINED in logistics export")
print("   Reason: essential for delivery routing; city alone (without name/phone)")
print("           does not identify an individual")
print()
print("Date: 2024-01-01  |  Analyst: [Your Name]  |  Reviewed by: [DPO Name]")

---
## Summary

In this demo we:
- Conducted a full PII audit of `orders_clean.csv` — classifying every column
- Applied name masking (initial only), phone masking (last 4 digits), email masking (domain only)
- Hashed `customer_name` to create a consistent, one-way `customer_id` pseudonym
- Built a privacy-safe export for a logistics partner — no direct PII, customer_id hash retained
- Verified the export contains no direct PII before saving
- Produced three fully aggregated public summaries (state, category, monthly)
- Applied the n<5 rule to suppress small groups
- Created a data file inventory and privacy decisions log

**The rule:** The right data, to the right people, with the right protections. Every time.

---
## Module 2 — Complete

You have worked through the full data pipeline:

| Topic | What we did |
|---|---|
| NumPy | Array operations on raw order amounts |
| Pandas | Loaded and inspected the full dataset |
| Cleaning | Fixed missing values, duplicates, types, and casing |
| Transforming | Filtered, grouped, aggregated, and merged |
| Statistics | Described distributions, detected outliers, measured correlation |
| Privacy | Anonymised PII and produced safe exports |

**Next — Module 3:** Exploratory Data Analysis & Visualisation.